In [1]:
import sys
from pathlib import Path
# Get project root (current working directory when running from project root)
# If running from research/, use parent; otherwise use cwd()
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

In [2]:
# Imports
from scripts.demo_ib_data_fetch import (
    IBDataClient, IBConfig, ContractSpec, MarketDataType, create_contract
)
import time
import datetime
import pandas as pd

# Date parser - handles daily bar format (YYYYMMDD) correctly
def parse_ib_date(date_str: str) -> pd.Timestamp:
    """Parse IB date string. Prioritizes YYYYMMDD format for daily bars."""
    if not date_str:
        return pd.NaT
    
    # Check YYYYMMDD format FIRST (daily bars)
    if len(date_str) == 8 and date_str.isdigit():
        try:
            year, month, day = int(date_str[:4]), int(date_str[4:6]), int(date_str[6:8])
            if 1970 <= year <= 2100 and 1 <= month <= 12 and 1 <= day <= 31:
                return pd.Timestamp(year, month, day)
        except (ValueError, IndexError):
            pass
    
    # Handle formats with time
    if ' ' in date_str or (len(date_str) > 8 and '-' in date_str):
        if ' ' in date_str:
            parts = date_str.split(' ', 2)
            if len(parts) >= 2 and len(parts[0]) == 8 and parts[0].isdigit():
                date_part, time_part = parts[0], parts[1]
                tz_part = parts[2] if len(parts) > 2 else None
                try:
                    dt_str = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]} {time_part}"
                    return pd.Timestamp(dt_str, tz=tz_part) if tz_part else pd.Timestamp(dt_str)
                except (ValueError, IndexError):
                    pass
        elif len(date_str) > 9 and date_str[8] == '-':
            try:
                date_part, time_part = date_str[:8], date_str[9:]
                if date_part.isdigit():
                    dt_str = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]} {time_part}"
                    return pd.Timestamp(dt_str, tz='UTC')
            except (ValueError, IndexError):
                pass
    
    # Epoch timestamp (10+ digits only)
    try:
        epoch = int(date_str)
        if len(date_str) >= 10:
            return pd.Timestamp.fromtimestamp(epoch / 1000 if epoch > 1e12 else epoch, tz='UTC')
    except (ValueError, OSError):
        pass
    
    try:
        return pd.Timestamp(date_str)
    except:
        return pd.NaT

# Interactive Brokers Data Fetching Experiments

This notebook allows you to experiment with the IB API data fetching functionality.

**Prerequisites:**
- TWS or IB Gateway must be running
- API settings enabled in TWS/Gateway
- Market data subscriptions (or use delayed data for testing)

In [3]:
# Configure IB connection
# Adjust port based on your setup:
# - TWS Paper: 7497
# - TWS Live: 7496
# - IB Gateway Paper: 4002
# - IB Gateway Live: 4001

config = IBConfig(
    host="127.0.0.1",
    port=7496,  # Change this if needed
    client_id=0
)

# Create client
client = IBDataClient(config)
print(f"Created IB client with config: {config}")

Created IB client with config: IBConfig(host='127.0.0.1', port=7496, client_id=0)


In [4]:
# Connect to IB
client.connect_to_ib()

# Wait for connection
max_wait = 5
waited = 0
while not client.connected and waited < max_wait:
    time.sleep(0.5)
    waited += 0.5

if client.connected:
    print("✓ Successfully connected to IB")
else:
    print("✗ Failed to connect. Check TWS/Gateway is running and API is enabled.")

Connecting to IB at 127.0.0.1:7496...
✓ Connected to IB. Next valid order ID: 1
✓ Successfully connected to IB



Head Timestamp (reqId: 2)
Epoch timestamp: 1639954800
Human readable: 2021-12-19 15:00:00
Bar (reqId: 3): Date: 20211220, Open: 4955.00, High: 4955.00, Low: 4955.00, Close: 4955.00, Volume: 0
Bar (reqId: 3): Date: 20211221, Open: 5042.00, High: 5042.00, Low: 5042.00, Close: 5042.00, Volume: 0
Bar (reqId: 3): Date: 20211222, Open: 5091.75, High: 5091.75, Low: 5091.75, Close: 5091.75, Volume: 0
Bar (reqId: 3): Date: 20211223, Open: 5126.50, High: 5126.50, Low: 5126.50, Close: 5126.50, Volume: 0
Bar (reqId: 3): Date: 20211227, Open: 5198.75, High: 5198.75, Low: 5198.75, Close: 5198.75, Volume: 0
Bar (reqId: 3): Date: 20211228, Open: 5199.75, High: 5199.75, Low: 5199.75, Close: 5199.75, Volume: 0
Bar (reqId: 3): Date: 20211229, Open: 5208.50, High: 5208.50, Low: 5208.50, Close: 5208.50, Volume: 0
Bar (reqId: 3): Date: 20211230, Open: 5193.25, High: 5193.25, Low: 5193.25, Close: 5193.25, Volume: 0
Bar (reqId: 3): Date: 20211231, Open: 5178.00, High: 5178.00, Low: 5178.00, Close: 5178.00, V

In [5]:
# Check max available history for NQ
contract = create_contract(ContractSpec("ES", "CONTFUT", "CME", "USD"))

# Request head timestamp (earliest available data)
client.request_head_timestamp(contract, what_to_show="TRADES", use_rth=0, format_date=2)

# Wait for response
time.sleep(2)

# Display result
if client.head_timestamp:
    ts = int(client.head_timestamp)
    dt = datetime.datetime.fromtimestamp(ts)
    print(f"✓ Earliest available data: {dt} (timestamp: {ts})")
    print(f"  Max history: ~{datetime.datetime.now().year - dt.year} years")
else:
    print("✗ No head timestamp received")

✓ Earliest available data: 2021-12-19 15:00:00 (timestamp: 1639954800)
  Max history: ~5 years


## Experiment 1: Request Head Timestamp

Find the earliest available data timestamp for a contract.

## Fetch NQ Historical Data

In [6]:
# Fetch NQ historical data
contract = create_contract(ContractSpec("ES", "CONTFUT", "CME", "USD"))

req_id = client.request_historical_data(
    contract, end_date_time="", duration="20 Y", bar_size="1 day",
    what_to_show="TRADES", use_rth=0, format_date=1, keep_up_to_date=False
)

print(f"Requested {req_id}, waiting...")
time.sleep(5)
print(f"Received {len(client.historical_bars)} bars")

Requested 3, waiting...
Received 1042 bars


In [ ]:
# Convert to DataFrame with fixed date parser
if client.historical_bars:
    df = pd.DataFrame([{
        'datetime': parse_ib_date(bar.date),
        'open': bar.open, 'high': bar.high, 'low': bar.low, 'close': bar.close,
        'volume': bar.volume,
        'wap': getattr(bar, 'average', None),
        'barCount': getattr(bar, 'barCount', None)
    } for bar in client.historical_bars])
    
    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)
    df['returns'] = df['close'].pct_change()
    df['range'] = df['high'] - df['low']
    df['body'] = abs(df['close'] - df['open'])
    
    print(f"✓ DataFrame: {df.shape[0]} bars, {df.index.min()} to {df.index.max()}")
    print(f"\nFirst 3:\n{df.head(3)[['open', 'high', 'low', 'close', 'volume']]}")
    print(f"\nLast 3:\n{df.tail(3)[['open', 'high', 'low', 'close', 'volume']]}")
    
    historical_df = df
else:
    print("No data")

In [ ]:
# Define contract to query
contract_spec = ContractSpec(
    symbol="QQQ",  # Try different symbols: "MSFT", "TSLA", "ES" (futures), etc.
    sec_type="STK",  # STK, FUT, OPT, CASH, etc.
    exchange="SMART",
    currency="USD"
)

contract = create_contract(contract_spec)
print(f"Contract: {contract_spec.symbol} ({contract_spec.sec_type})")

# Request head timestamp
client.request_head_timestamp(
    contract,
    what_to_show="TRADES",  # TRADES, MIDPOINT, BID, ASK
    use_rth=1,  # 1 = regular hours only, 0 = include extended hours
    format_date=2  # 2 = Epoch timestamp, 1 = UTC string
)

# Wait for response
time.sleep(2)

if client.head_timestamp:
    print(f"\nEarliest available data: {client.head_timestamp}")
    try:
        dt = datetime.datetime.fromtimestamp(int(client.head_timestamp))
        print(f"Human readable: {dt}")
    except (ValueError, OSError) as e:
        print(f"Could not convert: {e}")

## Get Today's Daily Bar

Fetch today's single daily bar (OHLC for the current trading day).

In [ ]:
# Define contract for NQ (E-mini Nasdaq futures) from CME
# For futures, we need to specify the contract month
contract_spec = ContractSpec(
    symbol="NQ",  # E-mini Nasdaq 100 futures
    sec_type="FUT",  # Futures
    exchange="CME",  # Chicago Mercantile Exchange (or "GLOBEX" for electronic)
    currency="USD"
)
contract = create_contract(contract_spec)

# Specify the contract month (March 2026 = MAR26)
# Format options: "202603" (YYYYMM) or "MAR26" (MMMYY)
contract.lastTradeDateOrContractMonth = "202603"  # March 2026
contract.tradingClass = "NQ"  # Trading class for NQ futures

# Request TODAY'S daily bar
# For today's bar, we request 1 day of daily bars ending at current time
end_time = ""  # Empty string = current time
duration = "1 D"  # 1 day duration
bar_size = "1 day"  # Daily bar
what_to_show = "TRADES"  # TRADES, MIDPOINT, BID, ASK
use_rth = 0  # 0 = include extended hours (futures trade nearly 24/7)

req_id = client.request_historical_data(
    contract,
    end_date_time=end_time,
    duration=duration,
    bar_size=bar_size,
    what_to_show=what_to_show,
    use_rth=use_rth,
    format_date=1,  # 1 = UTC string, 2 = Epoch
    keep_up_to_date=False  # Set to True if you want real-time updates
)

print(f"Requested today's daily bar with reqId: {req_id}")
print(f"Parameters: {duration} of {bar_size} bars, ending at {end_time or 'now'}")

# Wait for data
time.sleep(5)

# Check results
print(f"\n{'='*60}")
print(f"NQ Futures - Today's Daily Bar (reqId: {req_id})")
print(f"{'='*60}")
print(f"Received {len(client.historical_bars)} bar(s)")

if client.historical_bars:
    bar = client.historical_bars[0]  # Should only be one bar for today
    print(f"\nContract: NQ (E-mini Nasdaq 100 Futures)")
    print(f"Contract Month: MAR26 (March 2026)")
    print(f"Exchange: CME/GLOBEX")
    print(f"Date: {bar.date}")
    print(f"Open:  ${bar.open:.2f}")
    print(f"High:  ${bar.high:.2f}")
    print(f"Low:   ${bar.low:.2f}")
    print(f"Close: ${bar.close:.2f}")
    print(f"Volume: {bar.volume:,}")
    if hasattr(bar, 'average') and bar.average:
        print(f"VWAP: ${bar.average:.2f}")
    if hasattr(bar, 'barCount') and bar.barCount:
        print(f"Trade Count: {bar.barCount:,}")
    
    # Calculate daily change
    change = bar.close - bar.open
    change_pct = (change / bar.open) * 100
    print(f"\nDaily Change: ${change:+.2f} ({change_pct:+.2f}%)")
    print(f"Range: ${bar.high - bar.low:.2f} ({(bar.high - bar.low) / bar.open * 100:.2f}%)")
    print(f"\nNote: NQ futures trade nearly 24/7, so this bar includes extended hours trading.")
else:
    print("No data received.")
    print("Possible issues:")
    print("  - Market data subscription required for futures")
    print("  - Try using specific contract: sec_type='FUT' with lastTradeDateOrContractMonth='202603'")
    print("  - Or check if CME market data subscription is active")

## Alternative: Try Continuous Futures (CONTFUT) for NQ

Continuous Futures automatically uses the front month contract - easier than specifying contract months!

## Fixed Date Parser

Use this corrected version that properly handles daily bar dates (YYYYMMDD format).

In [ ]:


# Re-parse the historical bars with the fixed function
if client.historical_bars:
    bars_data = []
    for bar in client.historical_bars:
        bars_data.append({
            'datetime': parse_ib_date(bar.date),  # Use fixed parser
            'open': bar.open,
            'high': bar.high,
            'low': bar.low,
            'close': bar.close,
            'volume': bar.volume,
            'wap': bar.average if hasattr(bar, 'average') else None,
            'barCount': bar.barCount if hasattr(bar, 'barCount') else None
        })
    
    # Create DataFrame
    df = pd.DataFrame(bars_data)
    
    # Set datetime as index and sort
    if 'datetime' in df.columns:
        df.set_index('datetime', inplace=True)
        df.sort_index(inplace=True)
    
    # Add derived columns
    df['returns'] = df['close'].pct_change()
    df['range'] = df['high'] - df['low']
    df['body'] = abs(df['close'] - df['open'])
    
    print(f"\n{'='*60}")
    print(f"DataFrame Summary (Fixed Dates)")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]} bars × {df.shape[1]} columns")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"\nColumns: {', '.join(df.columns)}")
    
    print(f"\n{'='*60}")
    print("First few bars:")
    print(f"{'='*60}")
    print(df.head())
    
    print(f"\n{'='*60}")
    print("Last few bars:")
    print(f"{'='*60}")
    print(df.tail())
    
    print(f"\n{'='*60}")
    print("Basic Statistics:")
    print(f"{'='*60}")
    print(df[['open', 'high', 'low', 'close', 'volume']].describe())
    
    # Store in variable for further analysis
    historical_df = df
    print(f"\n✓ DataFrame stored in variable 'historical_df' with corrected dates")
else:
    print("No historical bars received.")

In [ ]:
# Try Continuous Futures (CONTFUT) - Automatically uses front month
# This is often easier than specifying contract months manually

contract_spec = ContractSpec(
    symbol="NQ",  # E-mini Nasdaq 100 futures
    sec_type="CONTFUT",  # Continuous Futures (auto-selects front month)
    exchange="CME",  # Chicago Mercantile Exchange
    currency="USD"
)
contract = create_contract(contract_spec)

# Request TODAY'S daily bar
# Note: For CONTFUT, endDateTime MUST be empty string
end_time = ""  # MUST be empty for CONTFUT
duration = "1 D"  # 1 day duration
bar_size = "1 day"  # Daily bar
what_to_show = "TRADES"  # TRADES, MIDPOINT, BID, ASK
use_rth = 0  # 0 = include extended hours (futures trade nearly 24/7)

req_id = client.request_historical_data(
    contract,
    end_date_time=end_time,  # Must be empty for CONTFUT
    duration=duration,
    bar_size=bar_size,
    what_to_show=what_to_show,
    use_rth=use_rth,
    format_date=1,  # 1 = UTC string, 2 = Epoch
    keep_up_to_date=False
)

print(f"Requested NQ Continuous Futures daily bar with reqId: {req_id}")
print(f"Parameters: {duration} of {bar_size} bars (CONTFUT uses front month automatically)")

# Wait for data
time.sleep(5)

# Check results
print(f"\n{'='*60}")
print(f"NQ Continuous Futures - Today's Daily Bar (reqId: {req_id})")
print(f"{'='*60}")
print(f"Received {len(client.historical_bars)} bar(s)")

if client.historical_bars:
    bar = client.historical_bars[0]
    print(f"\nContract: NQ (E-mini Nasdaq 100 Futures)")
    print(f"Type: Continuous Futures (CONTFUT) - Front Month Auto-Selected")
    print(f"Exchange: CME/GLOBEX")
    print(f"Date: {bar.date}")
    print(f"Open:  ${bar.open:.2f}")
    print(f"High:  ${bar.high:.2f}")
    print(f"Low:   ${bar.low:.2f}")
    print(f"Close: ${bar.close:.2f}")
    print(f"Volume: {bar.volume:,}")
    if hasattr(bar, 'average') and bar.average:
        print(f"VWAP: ${bar.average:.2f}")
    if hasattr(bar, 'barCount') and bar.barCount:
        print(f"Trade Count: {bar.barCount:,}")
    
    # Calculate daily change
    change = bar.close - bar.open
    change_pct = (change / bar.open) * 100
    print(f"\nDaily Change: ${change:+.2f} ({change_pct:+.2f}%)")
    print(f"Range: ${bar.high - bar.low:.2f} ({(bar.high - bar.low) / bar.open * 100:.2f}%)")
    print(f"\n✓ Successfully retrieved NQ futures daily bar!")
else:
    print("No data received.")
    print("\nIf CONTFUT doesn't work, try the specific contract month approach in the cell above.")

## Experiment 2: Request Historical Bar Data

Fetch historical OHLC bar data for a contract.

In [ ]:
# Define contract
contract_spec = ContractSpec(
    symbol="NQ",  # E-mini Nasdaq 100 futures
    sec_type="CONTFUT",  # Continuous Futures (auto-selects front month)
    exchange="CME",  # Chicago Mercantile Exchange
    currency="USD"
)
contract = create_contract(contract_spec)

# Request historical data
# You can customize these parameters:
end_time = ""  # Empty string = current time, or "YYYYMMDD HH:MM:SS TZ"
duration = "20 Y"  # "1 D", "1 W", "1 M", "1 Y"
bar_size = "1 day"  # "1 sec", "5 secs", "1 min", "1 hour", "1 day"
what_to_show = "TRADES"  # TRADES, MIDPOINT, BID, ASK
use_rth = 0  # 1 = regular hours only, 0 = include extended hours

req_id = client.request_historical_data(
    contract,
    end_date_time=end_time,
    duration=duration,
    bar_size=bar_size,
    what_to_show=what_to_show,
    use_rth=use_rth,
    format_date=1,  # 1 = UTC string, 2 = Epoch
    keep_up_to_date=False
)

print(f"Requested historical data with reqId: {req_id}")
print(f"Parameters: {duration} of {bar_size} bars, ending at {end_time or 'now'}")

# Wait for data
time.sleep(5)

# Check results
print(f"\nReceived {len(client.historical_bars)} bars")
if client.historical_bars:
    print(f"First bar: {client.historical_bars[0].date}, Close: {client.historical_bars[0].close:.2f}")
    print(f"Last bar: {client.historical_bars[-1].date}, Close: {client.historical_bars[-1].close:.2f}")

In [ ]:
# Convert historical bars to a more usable format with proper date parsing
import pandas as pd
from datetime import datetime
import re


if client.historical_bars:
    bars_data = []
    for bar in client.historical_bars:
        bars_data.append({
            'datetime': parse_ib_date(bar.date),
            'open': bar.open,
            'high': bar.high,
            'low': bar.low,
            'close': bar.close,
            'volume': bar.volume,
            'wap': bar.average if hasattr(bar, 'average') else None,  # Volume Weighted Average Price
            'barCount': bar.barCount if hasattr(bar, 'barCount') else None  # Number of trades
        })
    
    # Create DataFrame
    df = pd.DataFrame(bars_data)
    
    # Set datetime as index and sort
    if 'datetime' in df.columns:
        df.set_index('datetime', inplace=True)
        df.sort_index(inplace=True)
    
    # Add derived columns
    df['returns'] = df['close'].pct_change()
    df['range'] = df['high'] - df['low']
    df['body'] = abs(df['close'] - df['open'])
    
    print(f"\n{'='*60}")
    print(f"DataFrame Summary")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]} bars × {df.shape[1]} columns")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"\nColumns: {', '.join(df.columns)}")
    
    print(f"\n{'='*60}")
    print("First few bars:")
    print(f"{'='*60}")
    print(df.head())
    
    print(f"\n{'='*60}")
    print("Last few bars:")
    print(f"{'='*60}")
    print(df.tail())
    
    print(f"\n{'='*60}")
    print("Basic Statistics:")
    print(f"{'='*60}")
    print(df[['open', 'high', 'low', 'close', 'volume']].describe())
    
    # Store in variable for further analysis
    historical_df = df
    print(f"\n✓ DataFrame stored in variable 'historical_df' for further analysis")
else:
    print("No historical bars received. Check your request parameters and market data subscription.")

## Experiment 3: Request Live/Streaming Market Data

Subscribe to real-time or delayed market data ticks.

In [ ]:
# Set market data type
# Use DELAYED for testing without subscription, LIVE for real-time (requires subscription)
data_type = MarketDataType.DELAYED  # Change to MarketDataType.LIVE for real-time
client.set_market_data_type(data_type)
print(f"Market data type: {data_type.name}")

# Define contract
contract_spec = ContractSpec(
    symbol="NQ",  # E-mini Nasdaq 100 futures
    sec_type="CONTFUT",  # Continuous Futures (auto-selects front month)
    exchange="CME",  # Chicago Mercantile Exchange
    currency="USD"
)
contract = create_contract(contract_spec)

# Request market data
req_id = client.request_market_data(
    contract,
    generic_tick_list="232",  # "232" = Mark price, "232,233,234" = multiple ticks
    snapshot=True,  # True = single snapshot (last 11 seconds), False = streaming
    regulatory_snapshot=False  # True = regulatory snapshot (~$0.01 per request)
)

print(f"Subscribed to market data with reqId: {req_id}")
print("Streaming data for 10 seconds...\n")

# Stream for a bit
time.sleep(10)

# Cancel subscription
client.cancel_market_data(req_id)
print(f"\nCancelled market data subscription (reqId: {req_id})")

In [ ]:
# Cleanup: Disconnect when done
if client.connected:
    client.disconnect_from_ib()
    print("Disconnected from IB")